In [ ]:
import pycolmap
print('Is GPU Ready?:', pycolmap.has_cuda)

In [1]:
import torch
import pycolmap
import os

def check_environment():
    print("-" * 30)
    print("ENVIRONMENT CHECK FOR V100")
    print("-" * 30)

    # 1. Check PyTorch and CUDA
    torch_version = torch.__version__
    cuda_available = torch.cuda.is_available()
    
    print(f"PyTorch Version: {torch_version}")
    print(f"PyTorch CUDA Available: {cuda_available}")
    
    if cuda_available:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        bf16_support = torch.cuda.is_bf16_supported()
        print(f"GPU Name: {gpu_name}")
        print(f"GPU Memory: {gpu_mem:.2f} GB")
        print(f"BF16 Support: {bf16_support} (Recommended for DiT/5090 scripts)")
    else:
        print("ERROR: PyTorch cannot see the GPU!")

    print("-" * 30)

    # 2. Check PyColmap (3.13.0)
    colmap_version = pycolmap.__version__
    colmap_gpu = pycolmap.has_cuda
    
    print(f"PyColmap Version: {colmap_version}")
    print(f"PyColmap GPU Ready: {colmap_gpu}")
    
    if not colmap_gpu:
        print("WARNING: COLMAP is on CPU only. 1080p ginkgo reconstruction will be slow.")
    else:
        print("SUCCESS: COLMAP is ready for high-speed SiftGPU reconstruction.")

    print("-" * 30)

if __name__ == "__main__":
    check_environment()

------------------------------
ENVIRONMENT CHECK FOR V100
------------------------------
PyTorch Version: 2.10.0+cu128
PyTorch CUDA Available: True
GPU Name: Tesla V100-SXM2-32GB
GPU Memory: 16.00 GB
BF16 Support: True (Recommended for DiT/5090 scripts)
------------------------------
PyColmap Version: 4.0.2
PyColmap GPU Ready: True
SUCCESS: COLMAP is ready for high-speed SiftGPU reconstruction.
------------------------------


In [ ]:
import pycolmap

options = pycolmap.SiftExtractionOptions()

if hasattr(pycolmap, "has_cuda") and pycolmap.has_cuda:    
    print("SUCCESS: CUDA is available (library level)")
else:
    
    print("WARNING: Using CPU")

In [1]:
import cv2
import os
from pathlib import Path

# 1. Setup Paths
input_dir = Path("/home/aistudio/models/realityscan/Watt")
output_dir = Path("/home/aistudio/models/realityscan/Watt_Resized")
output_dir.mkdir(parents=True, exist_ok=True)

# 2. Process all .jpg files
print(f"Resizing JPGs in {input_dir}...")

for img_path in input_dir.glob("*.jpg"):
    img = cv2.imread(str(img_path))
    if img is None: continue
    
    # Calculate scale for 1024px on the longest side
    h, w = img.shape[:2]
    scale = 1024 / max(h, w)
    
    # Resize and Save
    resized = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(output_dir / img_path.name), resized)

print(f"Done! 400 images resized in: {output_dir}")

Resizing JPGs in /home/aistudio/models/realityscan/Watt...
Done! 400 images resized in: /home/aistudio/models/realityscan/Watt_Resized


In [5]:
import pycolmap
from pathlib import Path

# 1. Setup Absolute Paths
# Based on your root /home/aistudio

image_path =  Path("/home/aistudio/models/realityscan/Watt_Resized")
output_path =  Path("/home/aistudio/models/ColmapOutputs/Watt")
output_path.mkdir(parents=True, exist_ok=True)

database_folder = output_path / "database"
database_folder.mkdir(exist_ok=True)
database_path = database_folder / "database.db"

# Create output directories

sparse_path = output_path / "sparse"
sparse_path.mkdir(exist_ok=True)

reader_options = pycolmap.ImageReaderOptions()


sift_options = pycolmap.SiftExtractionOptions()

sift_options.max_num_features = 100 
# 2. Feature Extraction
# This identifies key points in your 400 images
print(f"Extracting features from: {image_path}")
pycolmap.extract_features(
    database_path=str(database_path),
    image_path=str(image_path),

)


Extracting features from: /home/aistudio/models/realityscan/Watt_Resized


I20260327 18:13:53.135923 140214173398784 feature_extraction.cc:502] === Feature extraction ===
I20260327 18:13:53.148951 140214048114432 sift.cc:754] Creating SIFT GPU feature extractor
I20260327 18:13:53.165953 140213334570752 feature_extraction.cc:280] Processed file [1/415]
I20260327 18:13:53.166017 140213334570752 feature_extraction.cc:283]   Name:            IMG_20260321_084301_1.jpg
W20260327 18:13:53.166022 140213334570752 feature_extraction.cc:287] IMG_20260321_084301_1.jpg IMAGE_EXISTS: Features for image were already extracted.
I20260327 18:13:53.166065 140213334570752 feature_extraction.cc:280] Processed file [2/415]
I20260327 18:13:53.166069 140213334570752 feature_extraction.cc:283]   Name:            IMG_20260321_084301_10.jpg
W20260327 18:13:53.166073 140213334570752 feature_extraction.cc:287] IMG_20260321_084301_10.jpg IMAGE_EXISTS: Features for image were already extracted.
I20260327 18:13:53.166079 140213334570752 feature_extraction.cc:280] Processed file [3/415]
I20

In [7]:



# 3. Feature Matching
# Exhaustive matching compares every image to every other image.
# For 400 images, this is thorough and reliable for 3D reconstruction.
print("Matching features...")
pycolmap.match_exhaustive(database_path=database_path)

# 4. Reconstruction (Structure from Motion)
# This calculates the 3D points and camera positions.
print("Starting incremental reconstruction...")
reconstructions = pycolmap.incremental_mapping(
    database_path=str(database_path),
    image_path=str(image_path),
    output_path=sparse_path
)

# 5. Finalize for Gaussian Splatting
# Gaussian Splatting usually expects the '0' folder (points3D.bin, cameras.bin, images.bin)
if reconstructions:
    print(f"Success! Found {len(reconstructions)} reconstruction clusters.")
    # Export the primary model to the '0' subfolder
    final_output = sparse_path / "0"
    final_output.mkdir(exist_ok=True)
    reconstructions[0].write(str(final_output))
    print(f"Model saved to: {final_output}")
else:
    print("Reconstruction failed. Check if your images have enough overlap.")

Matching features...


I20260327 18:14:38.478020 140214173398784 feature_matching.cc:195] === Feature matching & geometric verification ===
I20260327 18:14:38.478727 140213334570752 feature_matching_utils.cc:80] Bind FeatureMatcherWorker to GPU device 0
I20260327 18:14:38.480110 140213334570752 sift.cc:1547] Creating SIFT GPU feature matcher
I20260327 18:14:38.483221 140214173398784 pairing.cc:180] Generating exhaustive image pairs...
I20260327 18:14:38.483241 140214173398784 pairing.cc:213] Processing block [1/9, 1/9]
I20260327 18:15:05.402842 140214173398784 feature_matching.cc:217] in 26.920s
I20260327 18:15:05.402913 140214173398784 pairing.cc:213] Processing block [1/9, 2/9]
I20260327 18:15:28.267928 140214173398784 feature_matching.cc:217] in 22.865s
I20260327 18:15:28.268025 140214173398784 pairing.cc:213] Processing block [1/9, 3/9]
I20260327 18:15:53.150857 140214173398784 feature_matching.cc:217] in 24.883s
I20260327 18:15:53.150925 140214173398784 pairing.cc:213] Processing block [1/9, 4/9]
I20260

Starting incremental reconstruction...


I20260327 18:42:07.871448 140228666398528 database_cache.cc:139]  30579 in 0.203s
I20260327 18:42:07.871495 140228666398528 database_cache.cc:147] Loading images...
I20260327 18:42:08.147565 140228666398528 database_cache.cc:241]  415 in 0.276s (connected 415, loaded 415)
I20260327 18:42:08.147940 140228666398528 database_cache.cc:255] Loading pose priors...
I20260327 18:42:08.147965 140228666398528 database_cache.cc:263]  0 in 0.000s
I20260327 18:42:08.147973 140228666398528 database_cache.cc:271] Building correspondence graph...
I20260327 18:42:12.013310 140228666398528 database_cache.cc:300]  in 3.865s (ignored 0)
I20260327 18:42:12.014965 140228666398528 timer.cc:90] Elapsed time: 0.072 [minutes]
I20260327 18:42:12.087934 140228666398528 incremental_pipeline.cc:377] Finding good initial image pair
I20260327 18:42:15.252273 140228666398528 incremental_pipeline.cc:401] Registering initial image pair #38 and #246
I20260327 18:42:15.286894 140228666398528 incremental_pipeline.cc:419] G

Success! Found 1 reconstruction clusters.
Model saved to: /home/aistudio/models/ColmapOutputs/Watt/sparse/0


In [8]:
import pycolmap
from pathlib import Path

# 1. Setup paths
model_path = "/home/aistudio/models/ColmapOutputs/Watt/sparse/0"
output_ply = "/home/aistudio/models/ColmapOutputs/Watt/watt.ply"

# 2. Load the reconstruction
reconstruction = pycolmap.Reconstruction(model_path)

# 3. Export to PLY
print(f"Exporting {reconstruction.num_points3D()} points to PLY...")
reconstruction.export_PLY(output_ply)

print(f"Success! Your PLY is at: {output_ply}")

Exporting 108160 points to PLY...
Success! Your PLY is at: /home/aistudio/models/ColmapOutputs/Watt/watt.ply


In [1]:
import pycolmap
import numpy as np

# Path to your COLMAP sparse folder
sparse_folder = "/home/aistudio/models/ColmapOutputs/Watt/sparse/0"

# Load reconstruction
recon = pycolmap.Reconstruction(sparse_folder)

In [2]:
# recon.cameras is the mapping of CameraID -> Camera object
print(f"Number of cameras found: {len(recon.cameras)}")

for camera_id, camera in recon.cameras.items():
    print(f"\n--- Camera ID: {camera_id} ---")
    print(f"Model: {camera.model_name}")
    print(f"Dimensions: {camera.width} x {camera.height}")
    
    # These are the raw values from cameras.bin
    params = camera.params 
    print(f"Raw Parameters: {params}")

Number of cameras found: 415

--- Camera ID: 415 ---
Model: SIMPLE_RADIAL
Dimensions: 768 x 1024
Raw Parameters: [7.56080615e+02 3.84000000e+02 5.12000000e+02 3.23045351e-02]

--- Camera ID: 414 ---
Model: SIMPLE_RADIAL
Dimensions: 768 x 1024
Raw Parameters: [7.53108035e+02 3.84000000e+02 5.12000000e+02 2.86381245e-02]

--- Camera ID: 413 ---
Model: SIMPLE_RADIAL
Dimensions: 768 x 1024
Raw Parameters: [7.52338211e+02 3.84000000e+02 5.12000000e+02 3.35230315e-02]

--- Camera ID: 412 ---
Model: SIMPLE_RADIAL
Dimensions: 768 x 1024
Raw Parameters: [7.48186641e+02 3.84000000e+02 5.12000000e+02 3.93064756e-02]

--- Camera ID: 411 ---
Model: SIMPLE_RADIAL
Dimensions: 768 x 1024
Raw Parameters: [7.54272563e+02 3.84000000e+02 5.12000000e+02 1.37711221e-02]

--- Camera ID: 410 ---
Model: SIMPLE_RADIAL
Dimensions: 768 x 1024
Raw Parameters: [7.54681094e+02 3.84000000e+02 5.12000000e+02 1.14072099e-02]

--- Camera ID: 409 ---
Model: SIMPLE_RADIAL
Dimensions: 768 x 1024
Raw Parameters: [7.54393796

In [4]:
# points3D is a dict: {point3D_id: Point3D_object}
print(f"Total points in sparse cloud: {len(recon.points3D)}")

# Let's look at the first few points
for i, (point_id, point) in enumerate(recon.points3D.items()):
    if i >= 3: break
    
    xyz = point.xyz      # [x, y, z] coordinates
    rgb = point.color    # [r, g, b] (0-255)
    error = point.error  # Reprojection error (lower is better)
    
    print(f"Point {point_id}: Pos={xyz}, Color={rgb}, Error={error:.4f}")

Total points in sparse cloud: 108160
Point 140586: Pos=[ 10.40921443   6.00407386 -12.66443857], Color=[20 26 32], Error=1.4752
Point 140585: Pos=[ 0.39846907 -0.65832849  1.11374925], Color=[92 88 77], Error=1.8574
Point 140584: Pos=[ 0.21511483  5.3602956  -9.46664029], Color=[37 45 32], Error=1.9006


In [16]:
# Access image 67 specifically
img = recon.images[67]

print(f"{' Attribute ':=^30}")
for attr in dir(img):
    # Filter out internal/private methods (starting with __)
    
    try:
        value = getattr(img, attr)
        
        # If it's a method (like .summary()), call it to get the string
        if callable(value):
            print(f"{attr:20}: <Method>")
        else:
            print(f"{attr:20}: {value}")
    except Exception as e:
        print(f"{attr:20}: <Could not read: {e}>")



========= Attribute ==========
__class__           : <Method>
__copy__            : <Method>
__deepcopy__        : <Method>
__delattr__         : <Method>
__dir__             : <Method>
__doc__             : None
__eq__              : <Method>
__format__          : <Method>
__ge__              : <Method>
__getattribute__    : <Method>
__getstate__        : <Method>
__gt__              : <Method>
__hash__            : None
__init__            : <Method>
__init_subclass__   : <Method>
__le__              : <Method>
__lt__              : <Method>
__module__          : pycolmap._core
__ne__              : <Method>
__new__             : <Method>
__reduce__          : <Method>
__reduce_ex__       : <Method>
__repr__            : <Method>
__setattr__         : <Method>
__setstate__        : <Method>
__sizeof__          : <Method>
__str__             : <Method>
__subclasshook__    : <Method>
_pybind11_conduit_v1_: <Method>
cam_from_world      : <Method>
camera              : Camera(camera_id=6

In [20]:
# 1. Get the pose object
pose = img.cam_from_world()

# 2. Get the rotation (which is a Rotation3d object)
rotation = pose.rotation
trans = pose.translation

# 3. Get the quaternion [qw, qx, qy, qz]
quat = rotation.quat
print(f"Quaternion: {quat}")
print(f'translation: {trans}')

Quaternion: [ 0.00224863  0.89301363  0.16251089 -0.41965678]
translation: [ 0.22603547 -0.37753605  2.28470656]


In [13]:
for imgId,img in recon.images.items():
    print(imgId,img)

374 Image(image_id=374, camera_id=374, frame_id=374, name="IMG_20260321_084301_TIMEBURST61.jpg", has_pose=1, triangulated=535/8329)
167 Image(image_id=167, camera_id=167, frame_id=167, name="IMG_20260321_084301_61.jpg", has_pose=1, triangulated=535/8329)
373 Image(image_id=373, camera_id=373, frame_id=373, name="IMG_20260321_084301_TIMEBURST60.jpg", has_pose=1, triangulated=947/8213)
166 Image(image_id=166, camera_id=166, frame_id=166, name="IMG_20260321_084301_60.jpg", has_pose=1, triangulated=947/8213)
395 Image(image_id=395, camera_id=395, frame_id=395, name="IMG_20260321_084301_TIMEBURST80.jpg", has_pose=1, triangulated=1432/7560)
188 Image(image_id=188, camera_id=188, frame_id=188, name="IMG_20260321_084301_80.jpg", has_pose=1, triangulated=1432/7560)
67 Image(image_id=67, camera_id=67, frame_id=67, name="IMG_20260321_084301_159.jpg", has_pose=1, triangulated=1720/7492)
274 Image(image_id=274, camera_id=274, frame_id=274, name="IMG_20260321_084301_TIMEBURST159.jpg", has_pose=1, tr

In [7]:
for rig,rigInfo in recon.rigs.items():
    print(image,imageId)

415 Rig(rig_id=415, ref_sensor_id=(CAMERA, 415), sensors=[])
414 Rig(rig_id=414, ref_sensor_id=(CAMERA, 414), sensors=[])
413 Rig(rig_id=413, ref_sensor_id=(CAMERA, 413), sensors=[])
412 Rig(rig_id=412, ref_sensor_id=(CAMERA, 412), sensors=[])
411 Rig(rig_id=411, ref_sensor_id=(CAMERA, 411), sensors=[])
410 Rig(rig_id=410, ref_sensor_id=(CAMERA, 410), sensors=[])
409 Rig(rig_id=409, ref_sensor_id=(CAMERA, 409), sensors=[])
408 Rig(rig_id=408, ref_sensor_id=(CAMERA, 408), sensors=[])
407 Rig(rig_id=407, ref_sensor_id=(CAMERA, 407), sensors=[])
406 Rig(rig_id=406, ref_sensor_id=(CAMERA, 406), sensors=[])
405 Rig(rig_id=405, ref_sensor_id=(CAMERA, 405), sensors=[])
404 Rig(rig_id=404, ref_sensor_id=(CAMERA, 404), sensors=[])
403 Rig(rig_id=403, ref_sensor_id=(CAMERA, 403), sensors=[])
402 Rig(rig_id=402, ref_sensor_id=(CAMERA, 402), sensors=[])
401 Rig(rig_id=401, ref_sensor_id=(CAMERA, 401), sensors=[])
400 Rig(rig_id=400, ref_sensor_id=(CAMERA, 400), sensors=[])
399 Rig(rig_id=399, ref_

In [8]:
for frame,frameInfo in recon.frames.items():
    print(frame,frameInfo)

415 Frame(frame_id=415, rig_id=415, has_pose=1, data_ids=[(CAMERA, 415, 415)])
414 Frame(frame_id=414, rig_id=414, has_pose=1, data_ids=[(CAMERA, 414, 414)])
413 Frame(frame_id=413, rig_id=413, has_pose=1, data_ids=[(CAMERA, 413, 413)])
412 Frame(frame_id=412, rig_id=412, has_pose=1, data_ids=[(CAMERA, 412, 412)])
411 Frame(frame_id=411, rig_id=411, has_pose=1, data_ids=[(CAMERA, 411, 411)])
410 Frame(frame_id=410, rig_id=410, has_pose=1, data_ids=[(CAMERA, 410, 410)])
409 Frame(frame_id=409, rig_id=409, has_pose=1, data_ids=[(CAMERA, 409, 409)])
408 Frame(frame_id=408, rig_id=408, has_pose=1, data_ids=[(CAMERA, 408, 408)])
407 Frame(frame_id=407, rig_id=407, has_pose=1, data_ids=[(CAMERA, 407, 407)])
406 Frame(frame_id=406, rig_id=406, has_pose=1, data_ids=[(CAMERA, 406, 406)])
405 Frame(frame_id=405, rig_id=405, has_pose=1, data_ids=[(CAMERA, 405, 405)])
404 Frame(frame_id=404, rig_id=404, has_pose=1, data_ids=[(CAMERA, 404, 404)])
403 Frame(frame_id=403, rig_id=403, has_pose=1, data